# 16 · Drive test cache: one verified tar per test set (CPU runtime)

**CPU runtime** (no GPU). About **0.2–0.3 compute units** at 0.08 CU/h, for about 2–3 hours (estimate; the BR-Gen 7z extraction is the slow part).

**Why:** every GPU session so far spent a large share of its time staging test images (36 min in notebook 15). From Phase C on, a GPU notebook stages a test set **only by copying its tar from Drive and checking it**.

**What it writes**, in `Paper 1/runs/cache/tests/`, one pair per test set (`<name>.tar` and `<name>_manifest.json`):

| Name | Content |
|---|---|
| `p1_subset` | the frozen P1 subset (20,000 GenImage images) |
| `p2` | all `test_P2` DiffSeg30k images with their masks (2,434) |
| `p3_brgen` | the frozen BR-Gen subset (6,000 forged images with their masks) |
| `p3b` | the SID-Set `test_P3b` images (with masks where they exist) |
| `p4` | the frozen P4 Chameleon subset (5,000) |
| `claims_edit_src` | the COCO photos reserved for Claims-Edit (`test_P7`) |

- **Contents:** each tar holds the **original file bytes**, not re-encoded. Its manifest has the SHA-256 of every member and of the tar, plus each image's row (label, source, generator, mask base).
- **Checks after writing:** each tar is re-hashed **from Drive** and one seeded random member is decoded from the Drive copy. Both checks are recorded in the manifest.
- **Resume:** a set whose verified tar is already on Drive is skipped, so a dropped session restarts where it stopped.
- **P5** (about half of the spec's subset: 250 real + 250 generated per GenImage generator, plus 500 DiffSeg30k held-out images) is **frozen here** in `eval_subsets.json`. Its images are already in the P1 and P2 tars; the perturbations are applied at run time.
- **P6** already has its tar (`runs/cache/p6_images.tar`, notebook 15).

**Results** (send to Claude): `Paper 1/runs/cache/tests/test_cache_record.json` and `compute_test_cache.json`.

In [ ]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: fake sources, temp folders
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO + '[train]'], check=True)
    # Colab ships torchao 0.10; peft refuses a torchao older than 0.16 with an ImportError when it
    # adds the LoRA layers, although EviGround never uses torchao: remove an old one.
    import importlib, importlib.metadata as _md
    from packaging.version import Version
    try:
        if Version(_md.version('torchao')) < Version('0.16.0'):
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=True)
            importlib.invalidate_caches()
    except _md.PackageNotFoundError:
        pass

sys.path.insert(0, os.path.join(REPO, 'src'))
# Re-running in a runtime that already imported eviground would keep the OLD code in memory
# although git pulled the new one: drop it so the next import reads the files on disk.
for _name in [m for m in sys.modules if m == 'eviground' or m.startswith('eviground.')]:
    del sys.modules[_name]
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

In [ ]:
# Cell 2: settings
import json, logging, shutil, subprocess, tempfile, time, zipfile
from pathlib import Path
logging.basicConfig(level=logging.INFO, force=True)
from eviground.data import brgen, chameleon, testcache
from eviground.data.cp1 import _read_csv_gz
from eviground.env import get_paths, session_info
from eviground.eval import subsets, testsets
from eviground.utils import session

SEED = 42
EST_CU = 0.3               # CPU runtime: about 2-3 h at 0.08 CU/h (estimate)
CU_BALANCE_START = None    # optional: the balance Colab shows before Run all
T0 = time.time()
info = session_info(cu_per_hour=session.CU_RATES['cpu'])
paths = get_paths()
if DRY:
    from eviground.env import Paths
    _dry = Path(tempfile.gettempdir()) / 'eviground_testcache_dry'
    shutil.rmtree(_dry, ignore_errors=True)
    paths = Paths(root=_dry / 'drive' / 'Paper 1', local_data=_dry / 'data')
    EST_CU = 0.0
paths.ensure()
CK = paths.runs / 'checkpoints'
CP1_MANIFEST = CK / 'cp1' / 'manifest_cp1.csv.gz'
SUBSETS_FILE = CK / 'eval_subsets.json'
BRGEN_SUBSET = CK / 'brgen_subset.json'
BRGEN_ARCHIVE = paths.datasets / '04_BR-Gen' / brgen.LAST_VOLUME
CHAM_ZIP = paths.datasets / '05_Chameleon' / chameleon.ZIP_NAME
CACHE = paths.runs / 'cache' / 'tests'
BUILD = paths.local_data / 'testcache_build'
STAGE = paths.local_data / 'testcache_stage'
RECORD_FILE = CACHE / 'test_cache_record.json'
CACHE.mkdir(parents=True, exist_ok=True)
print('commit', info['git_commit'], '| gpu', info.get('gpu_name'), '| cache', CACHE)

In [ ]:
# Cell 3: in the dry run, a fake world with every input (CP1 manifest, frozen P1/P4, BR-Gen subset)
if DRY:
    import hashlib
    import numpy as np
    from eviground.data.cp1 import CP1Config, run_cp1
    from eviground.data.download import ensure_data
    from eviground.data.fake_cp1 import cp1_overrides, make_cp1_world
    from eviground.eval import p6data
    sources = make_cp1_world(paths.datasets, paths.root.parent / 'fake_remote', coco_images=40)
    ensure_data('cp1', paths=paths, sources=sources, stage_local=True, overrides=cp1_overrides(40))
    run_cp1(paths, CP1Config(workers=2, crops_per_class=60, blockiness_per_group=20, base_unclear_max=0.2,
                             diffseg_card_splits={'train': 18, 'validation': 12}),
            out_dir=CK / 'cp1', cache_dir=paths.cache / 'cp1', progress=None)
    rows = _read_csv_gz(CP1_MANIFEST)
    shutil.rmtree(paths.local_data / 'coco_val2017')   # a fresh runtime has no local COCO: cell 5 must stage it
    p1 = [r for r in rows if r['role'] == 'test_P1']
    rgb = np.full((60, 80, 3), 90, np.uint8); m = np.zeros((60, 80), np.uint8); m[10:30, 20:50] = 255
    BRGEN_ARCHIVE.parent.mkdir(parents=True, exist_ok=True); CHAM_ZIP.parent.mkdir(parents=True, exist_ok=True)
    forged, masks = [], {}
    with zipfile.ZipFile(BRGEN_ARCHIVE, 'w') as zf:
        for i in range(3):
            f, mk = f'BR-Gen/Forged/LaMa/Stuff/COCO/{i}_stuff.png', f'BR-Gen/Mask/Stuff/COCO/{i}_stuff.png'
            zf.writestr(f, p6data.png_bytes(rgb + i)); zf.writestr(mk, p6data.png_bytes(m))
            forged.append(f); masks[f] = mk
    from eviground.eval.devsplit import file_list_sha256
    BRGEN_SUBSET.write_text(json.dumps({'design': 'paired', 'items': sorted(forged), 'masks': masks,
                                        'sha256': file_list_sha256(sorted(forged))}))
    cham = ['Chameleon/test/0_real/r0.png', 'Chameleon/test/1_fake/f0.png']
    with zipfile.ZipFile(CHAM_ZIP, 'w') as zf:
        for c in cham:
            zf.writestr(c, p6data.png_bytes(rgb))
    subsets.save({'P1': subsets._record('P1', p1, SEED, 'dry'),
                  'P4': subsets._record('P4', [{'uid': c} for c in cham], SEED, 'dry')}, SUBSETS_FILE)
PREFLIGHT = session.preflight(
    {'notebook': '16 test cache (CPU)', 'cache folder': CACHE},
    paths={'CP1 manifest': CP1_MANIFEST, 'eval_subsets.json': SUBSETS_FILE, 'brgen_subset.json': BRGEN_SUBSET,
           'BR-Gen archive': BRGEN_ARCHIVE, 'Chameleon.zip': CHAM_ZIP},
    est_cu=EST_CU, cu_left=CU_BALANCE_START)
STORED = json.loads(SUBSETS_FILE.read_text())
for k in ('P1', 'P4'):
    assert subsets.intact(STORED.get(k, {})), f'{SUBSETS_FILE}: {k} is missing or was changed: stop and ask Dhanush'
BR = json.loads(BRGEN_SUBSET.read_text())
from eviground.eval.devsplit import file_list_sha256
assert file_list_sha256(BR['items']) == BR['sha256'], 'brgen_subset.json was changed: stop and ask Dhanush'
MANIFEST = _read_csv_gz(CP1_MANIFEST)
BY_UID = {r['uid']: r for r in MANIFEST}
print(f"manifest {len(MANIFEST)} rows | P1 {STORED['P1']['images']} | P4 {STORED['P4']['images']} | BR-Gen {len(BR['items'])}")

In [ ]:
# Cell 4: freeze P5 (about half the spec's subset) before any model sees it; reused if already frozen
P2_ROWS = [dict(r) for r in MANIFEST if r.get('role') == 'test_P2']
if 'P5' in STORED:
    assert subsets.intact(STORED['P5']), 'P5 in eval_subsets.json was changed: stop and ask Dhanush'
    P5 = STORED['P5']; P5_STATUS = 'reused'
else:
    p1_rows = [BY_UID[u] for u in STORED['P1']['uids']]
    kw = {'per_label': 1, 'diffseg': 1} if DRY else {}
    P5 = subsets.p5_subset(p1_rows, P2_ROWS, **kw)
    STORED['P5'] = P5
    subsets.save(STORED, SUBSETS_FILE)
    P5_STATUS = 'frozen now'
print(f"P5 {P5_STATUS}: {P5['images']} images, SHA-256 {P5['sha256'][:12]} "
      f"({P5['diffseg_heldout']} DiffSeg30k held-out; per stratum {sorted(set(P5['per_stratum'].values()))})")

In [ ]:
# Cell 5: one tar per test set: stage, build, publish (Drive re-hash + one decode), then free the disk
RECORD = {'sets': {}, 'P5': {k: P5[k] for k in ('images', 'sha256', 'seed', 'salt', 'diffseg_heldout')}}
_dataset_staged = set()

def _ensure(datasets):
    # stage each dataset a set reads from the local disk (manifest paths point at the local copies)
    todo = [d for d in datasets if d not in _dataset_staged]
    if todo:
        from eviground.data.download import ensure_data
        kw = {'sources': sources, 'overrides': cp1_overrides(40)} if DRY else {}
        ensure_data(datasets=todo, paths=paths, stage_local=True, **kw)
    _dataset_staged.update(todo)

def rows_p1():
    return testsets.stage_files([BY_UID[u] for u in STORED['P1']['uids']], STAGE / 'p1_subset')

def rows_p2():
    _ensure(['diffseg30k'])
    return P2_ROWS

def rows_p3b():
    _ensure(['sidset'])
    return [dict(r) for r in MANIFEST if r.get('role') == 'test_P3b']

def rows_brgen():
    exe = brgen.seven_zip()
    if exe is None and not DRY:
        subprocess.run(['apt-get', 'install', '-y', '-q', 'p7zip-full'], check=True, capture_output=True)
        exe = brgen.seven_zip()
    assert exe, '7z is not available'
    out = STAGE / 'p3_brgen'
    members = list(BR['items']) + sorted(set(BR['masks'].values()))
    brgen.extract(BRGEN_ARCHIVE, members, out, exe)
    return [{'uid': f, 'source': 'brgen', 'label': 'edited', 'path': str(out / f), 'mask_path': str(out / BR['masks'][f])}
            for f in BR['items']]

def rows_p4():
    out = STAGE / 'p4'
    with zipfile.ZipFile(CHAM_ZIP) as zf:
        for u in STORED['P4']['uids']:
            zf.extract(u, out)
    return [{'uid': u, 'source': 'chameleon', 'label': chameleon.label_of(u), 'path': str(out / u)}
            for u in STORED['P4']['uids']]

def rows_claims():
    _ensure(['coco_val2017'])
    return testsets.stage_files([dict(r) for r in MANIFEST if r.get('role') == 'test_P7'], STAGE / 'claims_edit_src')

SETS = {'p1_subset': rows_p1, 'p2': rows_p2, 'p3_brgen': rows_brgen, 'p3b': rows_p3b, 'p4': rows_p4,
        'claims_edit_src': rows_claims}
for name, fn in SETS.items():
    t = time.time()
    if testcache.is_ready(CACHE, name):
        man = json.loads(testcache.paths(CACHE, name)[1].read_text()); status = 'reused'
    else:
        rows = fn()
        man = testcache.publish(testcache.build(rows, name, BUILD), BUILD, CACHE, seed=SEED)
        for p in testcache.paths(BUILD, name):
            p.unlink(missing_ok=True)
        shutil.rmtree(STAGE / name, ignore_errors=True)
        status = 'built'
    RECORD['sets'][name] = {'status': status, 'images': man['images'], 'masks': man['masks'],
                            'tar_bytes': man['tar_bytes'], 'tar_sha256': man['tar_sha256'],
                            'drive_sha256_ok': man.get('drive_sha256_ok'), 'decode_check': man.get('decode_check'),
                            'minutes': round((time.time() - t) / 60, 2)}
    r = RECORD['sets'][name]
    print(f"{name:16} {status}: {r['images']} images, {r['masks']} masks, {r['tar_bytes'] / 1e9:.2f} GB, "
          f"tar {r['tar_sha256'][:12]}, Drive re-hash {r['drive_sha256_ok']}, decoded {r['decode_check']} | {r['minutes']} min")

In [ ]:
# Cell 6: optional, no GPU: CP5 per part as run and without the 26 aspect-mismatched SID-Set masks
# (a sensitivity check only; the CP5 record stays as run). Skipped if the notebook 14/15 files are not on Drive.
import csv
CP5_DIR = paths.runs / 'cp5'
F14 = CP5_DIR / f'p6_faithfulness_seed{SEED}.csv'
F15 = CP5_DIR / f'p6_token_deletion_records_seed{SEED}.jsonl'
R15 = CP5_DIR / f'cp5_token_deletion_seed{SEED}.json'
if all(f.is_file() for f in (F14, F15, R15)):
    from eviground.explain import cp5
    with open(F14, newline='') as fh:
        p6_rows = list(csv.DictReader(fh))
    tok_rows = [json.loads(line) for line in F15.read_text().splitlines() if line.strip()]
    off = [a['uid'] for a in json.loads(R15.read_text())['p6_mask_resize']['parts'].get('_aspect_off', [])]
    P6_MAN, P6_TAR = paths.runs / 'cache' / 'p6_images_manifest.json', paths.runs / 'cache' / 'p6_images.tar'
    if not off and P6_MAN.is_file() and P6_TAR.is_file():   # fallback: test each resized SID-Set mask against its image
        import io, tarfile
        from PIL import Image
        from eviground.eval import p6data
        man = json.loads(P6_MAN.read_text())
        with tarfile.open(P6_TAR) as tf:
            for uid, rec in man['entries'].items():
                meta = rec['meta']
                if meta.get('part') == 'sidset_tampered' and meta.get('mask_resized_from'):
                    with Image.open(io.BytesIO(tf.extractfile(rec['image']).read())) as im:
                        if p6data.is_aspect_mismatched(meta, (im.height, im.width)):
                            off.append(uid)
    resized_sid = sum(1 for r in tok_rows if r.get('part') == 'sidset_tampered') and json.loads(R15.read_text())[
        'p6_mask_resize']['parts'].get('sidset_tampered', {}).get('resized', 0)
    assert off or not resized_sid, 'SID-Set masks were resized but no aspect-mismatched uid was found: stop and ask Dhanush'
    print(f'aspect-mismatched SID-Set uids left out in the sensitivity variant: {len(off)}')
    SENS = cp5.aspect_sensitivity(p6_rows, tok_rows, off)
    for part, row in SENS['parts'].items():
        for variant, r in row.items():
            epg = {k: (None if v is None else round(v, 3)) for k, v in r['epg_ratio'].items()}
            print(f"{part:16} {variant:16} EPG ratio {epg} ({r['edited_images']} images) | token deletion "
                  f"valid {r['token_valid']}, {r['token_maps']} ({r['token_images']} images)")
else:
    SENS = None
    print('CP5 sensitivity skipped: notebook 14/15 result files are not on Drive')

In [ ]:
# Cell 7: the record, a staging check, the compute record, then release the runtime
import datetime
check = testcache.stage(CACHE, 'p2', paths.local_data / 'testcache_check')   # one full round trip
assert len(check) == RECORD['sets']['p2']['images']
items = testsets.file_items(check[:2])
RECORD['stage_check'] = {'set': 'p2', 'rows': len(check), 'first_items_decoded': [it.image.shape for it in items]}
RECORD['cp5_aspect_sensitivity'] = SENS
RECORD.update(dry_run=DRY, git_commit=info['git_commit'],
              utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'))
RECORD_FILE.write_text(json.dumps(RECORD, indent=1, default=str))
print('saved', RECORD_FILE)
COMPUTE = session.finish(CACHE, T0, gpu=info.get('gpu_name') or 'cpu', name='compute_test_cache.json',
                         balance_start=CU_BALANCE_START, extra={'git_commit': info['git_commit'], 'est_cu': EST_CU})
if DRY:
    assert all(r['drive_sha256_ok'] for r in RECORD['sets'].values())
    assert RECORD['sets']['p3_brgen']['masks'] == RECORD['sets']['p3_brgen']['images'] == 3
RELEASED = session.release(DRY)
if DRY:
    assert RELEASED == 'dry', RELEASED